**Mô tả Notebook**  
Notebook này **chuyển đổi cấu trúc thư mục chứa ảnh và file nhãn json thành định dạng Hugging Face Dataset chuẩn (gồm đối tượng image và text)** để làm input trực tiếp cho mô hình DeepSeek-OCR.

In [1]:
# %% [1] Cài đặt thư viện cần thiết
!pip install datasets pandas pillow

In [2]:
# %% [2] Import thư viện
import os
import json
import pandas as pd
from datasets import Dataset, DatasetDict, Image, Features, Value

In [3]:
# %% [3] Hàm xử lý chính (Cập nhật thêm cột 'type')
ROOT_DIR = "/kaggle/input/uit-hwdb/UIT_HWDB" 

def get_image_paths_and_labels_strict(root_dir):
    train_data = []
    test_data = []
    
    total_valid_pairs = 0

    print(f"🚀 Bắt đầu quét dữ liệu từ: {root_dir}")
    if not os.path.exists(root_dir):
        print("❌ LỖI: Đường dẫn ROOT_DIR không tồn tại!")
        return [], []

    print("-" * 50)
    
    # Duyệt qua cây thư mục
    for dirpath, dirnames, filenames in os.walk(root_dir):
        if "label.json" in filenames:
            json_path = os.path.join(dirpath, "label.json")
            
            # 1. Chuẩn hóa đường dẫn
            norm_path = dirpath.replace("\\", "/")
            path_parts = norm_path.split("/")
            
            # 2. Xác định Train/Test
            split_type = None
            for part in path_parts:
                part_lower = part.lower()
                if 'train' in part_lower: 
                    split_type = 'train'
                    break
                elif 'test' in part_lower:
                    split_type = 'test'
                    break
            
            if split_type is None: continue 

            # 3. Xác định Type (word, line, paragraph) 
            # Logic: Tìm keyword trong đường dẫn
            img_type = "unknown"
            path_lower = norm_path.lower()
            
            if "word" in path_lower:
                img_type = "word"
            elif "line" in path_lower:
                img_type = "line"
            elif "paragraph" in path_lower: 
                img_type = "paragraph"
            
            # Đọc file JSON
            try:
                with open(json_path, 'r', encoding='utf-8') as f:
                    labels = json.load(f)
            except Exception as e:
                print(f"❌ Lỗi đọc file {json_path}: {e}")
                continue

            count_on_disk = len([f for f in filenames if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
            added_count = 0
            
            for img_name, text_label in labels.items():
                full_img_path = os.path.join(dirpath, img_name)
                
                if os.path.exists(full_img_path):
                    # Thêm trường 'type' vào entry
                    entry = {
                        "image": full_img_path,
                        "text": text_label,
                        "type": img_type 
                    }
                    
                    if split_type == 'train':
                        train_data.append(entry)
                    else:
                        test_data.append(entry)
                    added_count += 1
            
            total_valid_pairs += added_count
            
            if added_count > 0:
                # In thêm Type để debug
                print(f"✅ [{img_type.upper()}] .../{os.path.basename(dirpath)} -> {split_type.upper()} | Lấy: {added_count} ảnh")

    print("-" * 50)
    print(f"🎉 Hoàn tất! Tổng số mẫu hợp lệ: {total_valid_pairs}")
    print(f"   - Train: {len(train_data)}")
    print(f"   - Test: {len(test_data)}")
    
    return train_data, test_data

In [4]:
# %% [4] Thực thi hàm quét dữ liệu
train_list, test_list = get_image_paths_and_labels_strict(ROOT_DIR)

🚀 Bắt đầu quét dữ liệu từ: /kaggle/input/uit-hwdb/UIT_HWDB
--------------------------------------------------
✅ [PARAGRAPH] .../253 -> TEST | Lấy: 7 ảnh
✅ [PARAGRAPH] .../250 -> TEST | Lấy: 5 ảnh
✅ [PARAGRAPH] .../254 -> TEST | Lấy: 3 ảnh
✅ [PARAGRAPH] .../251 -> TEST | Lấy: 6 ảnh
✅ [PARAGRAPH] .../255 -> TEST | Lấy: 4 ảnh
✅ [PARAGRAPH] .../252 -> TEST | Lấy: 6 ảnh
✅ [PARAGRAPH] .../248 -> TRAIN | Lấy: 5 ảnh
✅ [PARAGRAPH] .../7 -> TRAIN | Lấy: 6 ảnh
✅ [PARAGRAPH] .../135 -> TRAIN | Lấy: 5 ảnh
✅ [PARAGRAPH] .../47 -> TRAIN | Lấy: 6 ảnh
✅ [PARAGRAPH] .../183 -> TRAIN | Lấy: 7 ảnh
✅ [PARAGRAPH] .../17 -> TRAIN | Lấy: 3 ảnh
✅ [PARAGRAPH] .../81 -> TRAIN | Lấy: 2 ảnh
✅ [PARAGRAPH] .../19 -> TRAIN | Lấy: 6 ảnh
✅ [PARAGRAPH] .../199 -> TRAIN | Lấy: 4 ảnh
✅ [PARAGRAPH] .../121 -> TRAIN | Lấy: 6 ảnh
✅ [PARAGRAPH] .../192 -> TRAIN | Lấy: 5 ảnh
✅ [PARAGRAPH] .../22 -> TRAIN | Lấy: 7 ảnh
✅ [PARAGRAPH] .../2 -> TRAIN | Lấy: 5 ảnh
✅ [PARAGRAPH] .../229 -> TRAIN | Lấy: 5 ảnh
✅ [PARAGRAPH] .../240 -> 

In [5]:
# %% [5] Nén file Zip
import shutil
import os

if len(train_list) > 0:
    # 1. Chuyển list thành DataFrame
    df_train = pd.DataFrame(train_list)
    df_test = pd.DataFrame(test_list)

    # 2. Định nghĩa Features (Bao gồm cột type mới)
    features = Features({
        'image': Image(),
        'text': Value('string'),
        'type': Value('string') 
    })

    # 3. Tạo Dataset
    print("⏳ Đang tạo Dataset object...")
    ds_train = Dataset.from_pandas(df_train, features=features)
    
    if len(test_list) > 0:
        ds_test = Dataset.from_pandas(df_test, features=features)
    else:
        ds_test = Dataset.from_dict({"image": [], "text": [], "type": []}, features=features)

    # 4. Gộp thành DatasetDict
    final_dataset = DatasetDict({
        "train": ds_train,
        "test": ds_test
    })

    print("\nDataset Structure:")
    print(final_dataset)

    # 5. Lưu folder dataset ra đĩa
    save_folder = "uit_hwdb_processed"
    final_dataset.save_to_disk(save_folder)
    print(f"✅ Đã lưu dataset sạch vào folder '{save_folder}'")

    # 6. --- NÉN ZIP ĐỂ TẢI VỀ ---
    output_zip_name = "DATA_PROCESSED"
    print(f"📦 Đang nén folder '{save_folder}' thành file zip...")
    
    # shutil.make_archive(tên_file_zip, định_dạng, folder_cần_nén)
    shutil.make_archive(output_zip_name, 'zip', save_folder)
    
    print(f"🎉 HOÀN TẤT! File nén đã sẵn sàng: {output_zip_name}.zip")
    print(f"👉 Hãy tải file '{output_zip_name}.zip' về máy và upload lên Kaggle Dataset.")

else:
    print("❌ Không tìm thấy dữ liệu nào hợp lệ.")

⏳ Đang tạo Dataset object...

Dataset Structure:
DatasetDict({
    train: Dataset({
        features: ['image', 'text', 'type'],
        num_rows: 115748
    })
    test: Dataset({
        features: ['image', 'text', 'type'],
        num_rows: 3113
    })
})


Saving the dataset (0/108 shards):   0%|          | 0/115748 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3113 [00:00<?, ? examples/s]

✅ Đã lưu dataset sạch vào folder 'uit_hwdb_processed'
📦 Đang nén folder 'uit_hwdb_processed' thành file zip...
🎉 HOÀN TẤT! File nén đã sẵn sàng: DATA_PROCESSED.zip
👉 Hãy tải file 'DATA_PROCESSED.zip' về máy và upload lên Kaggle Dataset.
